# Does defect-only data bias a detector toward over-rejection?

`overkill-and-prevalence.md` showed that on NEU-DET — where **every image contains a
defect** — lowering the decision threshold looks free, because the overkill term is
multiplied by (1−p) = 0. The overkill rate there had to be estimated from defect-free
*regions*, giving an upper bound rather than a measurement.

**KolektorSDD2 has real defect-free images**: 894 normal against 110 defective in test,
a natural prevalence of **11%**. So both error rates can be measured directly, and the
break-even analysis can be redone without a proxy.

It also allows a stronger test. Two arms, identical except for the training data:

| arm | trained on | mimics |
|---|---|---|
| **defect-only** | the 246 defective train images | NEU-DET's regime |
| **with-normals** | those plus 2085 normal images as backgrounds | a real line |

If the defect-only arm shows a much higher overkill rate at the same escape rate, then
the bias is not only in threshold *selection* — it is baked in at training time by the
composition of the benchmark.

Single class (`defect`), which matches the PASS/FAIL decision the system actually makes.
**Session options → Accelerator → GPU T4 x2.**


In [ ]:
import torch, os
print('torch', torch.__version__)
assert torch.cuda.is_available(), 'No GPU. Session options > Accelerator > GPU T4 x2.'
print(torch.cuda.get_device_name(0))
%pip -q install ultralytics
import ultralytics; print('ultralytics', ultralytics.__version__)


## Build YOLO labels from the segmentation masks

Each `NNNNN.png` has an `NNNNN_GT.png` mask. Connected components become boxes. An
all-zero mask means a genuinely defect-free image — written as an **empty label file**,
which is how YOLO represents a background image. Those are the images NEU-DET does not
have.


In [ ]:
import glob, shutil, cv2, numpy as np
from pathlib import Path

# Kaggle mounts a dataset under a directory whose name is not always the slug
# (this one arrives as /kaggle/input/datasets). Find it by structure instead of
# by name: the directory that contains train/ and test/ full of PNGs.
def find_root(base=Path('/kaggle/input')):
    for cand in [base, *base.rglob('*')]:
        if not cand.is_dir():
            continue
        if (cand/'train').is_dir() and (cand/'test').is_dir():
            if any((cand/'train').glob('*.png')):
                return cand
    raise SystemExit('KSDD2 not found under /kaggle/input; saw: '
                     + str(sorted(str(x.relative_to(base)) for x in base.rglob('*') if x.is_dir())[:40]))

SRC = find_root()
print('dataset root:', SRC)

def boxes_from_mask(mask):
    n, _, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    out = []
    for i in range(1, n):
        x, y, w, h, area = stats[i]
        if area < 4:            # a few stray pixels are annotation noise, not a defect
            continue
        out.append((x, y, w, h))
    return out

INDEX = {}   # split -> list of (stem, path, boxes, is_defective)
for split in ('train', 'test'):
    rows = []
    for img_path in sorted(SRC.glob(f'{split}/*.png')):
        if img_path.stem.endswith('_GT'):
            continue
        gt = img_path.with_name(img_path.stem + '_GT.png')
        if not gt.exists():
            continue
        m = cv2.imread(str(gt), cv2.IMREAD_GRAYSCALE)
        bs = boxes_from_mask(m) if m is not None and m.max() > 0 else []
        rows.append((img_path.stem, img_path, bs, len(bs) > 0))
    INDEX[split] = rows
    d = sum(1 for r in rows if r[3])
    print(f'{split}: {len(rows)} images, {d} defective ({d/len(rows)*100:.1f}%), {len(rows)-d} normal')

assert INDEX['test'], 'no test images found'


In [ ]:
def build(name, include_normals):
    """Write a YOLO tree. Test split is ALWAYS the full one -- both arms are
    judged on the same data, including the normal images."""
    root = Path(f'/kaggle/working/ds_{name}')
    if root.exists(): shutil.rmtree(root)
    counts = {}
    for split in ('train', 'test'):
        (root/split/'images').mkdir(parents=True, exist_ok=True)
        (root/split/'labels').mkdir(parents=True, exist_ok=True)
        n = 0
        for stem, path, bs, is_def in INDEX[split]:
            if split == 'train' and not include_normals and not is_def:
                continue
            im = cv2.imread(str(path)); H, W = im.shape[:2]
            cv2.imwrite(str(root/split/'images'/f'{stem}.jpg'), im)
            lines = [f'0 {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}'
                     for x, y, w, h in bs]
            (root/split/'labels'/f'{stem}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
            n += 1
        counts[split] = n
    (root/'data.yaml').write_text('train: train/images\nval: test/images\n'
                                 'test: test/images\nnames:\n  0: defect\n')
    print(f'{name}: train {counts["train"]}, test {counts["test"]}')
    return root

DS = {'defect_only': build('defect_only', False),
      'with_normals': build('with_normals', True)}


## Train both arms

Identical settings. 640 px because KSDD2 defects are small — median 1.65% of image area,
against 39% on NEU-DET.


In [ ]:
from ultralytics import YOLO
import time
W = {}
for name, root in DS.items():
    t0 = time.time()
    m = YOLO('yolov8n.pt')
    m.train(data=str(root/'data.yaml'), imgsz=640, epochs=60, patience=20, seed=42,
            deterministic=True, device=0, project='/kaggle/working/runs', name=name,
            exist_ok=True, verbose=False)
    W[name] = f'/kaggle/working/runs/{name}/weights/best.pt'
    print(f'{name}: {(time.time()-t0)/60:.1f} min')


## Measure both error rates directly

**escape** = a defective image with no detection → it ships.
**overkill** = a genuinely defect-free image that fires → a good part is scrapped.

No proxy, no crops, no upper bound. `rect=False` throughout, per
`reports/imgsz-rect-trap.md`.


In [ ]:
import numpy as np, pandas as pd
THRS = [0.50, 0.35, 0.25, 0.15, 0.10, 0.05, 0.01]

test_def = [r for r in INDEX['test'] if r[3]]
test_nor = [r for r in INDEX['test'] if not r[3]]
print(f'{len(test_def)} defective and {len(test_nor)} normal test images')

rows = []
for name, w in W.items():
    m = YOLO(w)
    def maxconf(items):
        out = []
        for _, path, _, _ in items:
            r = m.predict(str(path), imgsz=640, conf=0.01, rect=False, verbose=False)[0]
            c = r.boxes.conf.cpu().numpy() if len(r.boxes) else np.empty(0)
            out.append(float(c.max()) if len(c) else 0.0)
        return np.array(out)
    cd, cn = maxconf(test_def), maxconf(test_nor)
    for t in THRS:
        rows.append({'arm': name, 'thr': t,
                     'escape': float((cd < t).mean()),
                     'overkill': float((cn >= t).mean())})
    print(f'{name}: done')

df = pd.DataFrame(rows)
df.to_csv('/kaggle/working/ksdd2_rates.csv', index=False)
print()
print('ESCAPE (defective image passes silently)')
print(df.pivot(index='thr', columns='arm', values='escape').to_string())
print()
print('OVERKILL (defect-free image rejected) -- measured, not estimated')
print(df.pivot(index='thr', columns='arm', values='overkill').to_string())


## The economics, at a real prevalence

KSDD2's own test prevalence is **11%**. NEU-DET's is 100%, which is why the overkill
term vanished there.


In [ ]:
P_NAT = len(test_def)/(len(test_def)+len(test_nor))
print(f'natural prevalence of this dataset: {P_NAT:.3f}\n')
out = []
for arm in df['arm'].unique():
    sub = df[df.arm == arm].set_index('thr')
    for p in (P_NAT, 0.01, 0.05, 0.50, 1.00):
        for R in (1, 10, 100, 1000):
            cost = {t: p*sub.loc[t,'escape']*R + (1-p)*sub.loc[t,'overkill'] for t in THRS}
            out.append({'arm': arm, 'prevalence': round(p,3), 'R': R,
                        'best_thr': min(cost, key=cost.get)})
o = pd.DataFrame(out)
o.to_csv('/kaggle/working/ksdd2_operating_points.csv', index=False)
for arm in o['arm'].unique():
    print(f'--- {arm}: cost-optimal threshold ---')
    print(o[o.arm==arm].pivot(index='prevalence', columns='R', values='best_thr').to_string())
    print()
print('prevalence 1.00 is the NEU-DET regime: the overkill term is multiplied by zero.')


### What to read

1. **Does the defect-only arm overkill more** at matched escape rate? If yes, benchmark
   composition biases the model itself, not merely the threshold chosen afterwards.
2. **Does the optimal threshold move with prevalence?** The row at prevalence 1.00 should
   recommend a far lower threshold than the row at 0.11 — that difference *is* the
   NEU-DET artefact, reproduced on a dataset that does not have it.
3. Both arms are judged on the **same** test set including normals, so the comparison is
   about training composition only.


In [ ]:
import shutil, os
for n in DS: shutil.copy(W[n], f'/kaggle/working/best_{n}.pt')
for n in DS:
    d=f'/kaggle/working/runs/{n}'
    if os.path.exists(d):
        shutil.copytree(d, f'/kaggle/working/{n}_run', dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('*.pt'))
    shutil.rmtree(f'/kaggle/working/ds_{n}', ignore_errors=True)
shutil.rmtree('/kaggle/working/runs', ignore_errors=True)
print(sorted(os.listdir('/kaggle/working')))
